<a href="https://colab.research.google.com/github/0517kkm-blip/-/blob/main/%ED%95%B4%EC%82%AC%EB%8D%B0%EC%9D%B4%ED%84%B0%EB%A7%88%EC%9D%B4%EB%8B%9D%209%EB%B2%88%EC%A7%B8%20%EA%B3%BC%EC%A0%9C%2020230525%20%EA%B9%80%EA%B7%9C%EB%AF%BC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import os
import zipfile
from pathlib import Path
import pandas as pd
from scipy.stats import chi2_contingency

# 압축 해제 및 경로 자동 설정
zip_path = 'type3_sample_csvs.zip'
if os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall('/content/type3')

# 파일 위치 자동 검색 (이중 폴더 문제 해결)
file_path = None
for root, dirs, files in os.walk('/content/type3'):
    if 'channel_purchase.csv' in files:
        file_path = os.path.join(root, 'channel_purchase.csv')
        break

# 1. 데이터 불러오기
df = pd.read_csv(file_path)

# 2. 데이터 확인
print('[데이터 상위 5행]')
print(df.head(), '\n')

print('[변수 정보]')
print(df.info(), '\n')

# 3. 교차표 생성
table = pd.crosstab(df['channel'], df['purchase_yn'])

print('[교차표]')
print(table, '\n')

# 4. 카이제곱 독립성 검정
chi2, p, dof, expected = chi2_contingency(table)

print('[검정 결과]')
print('chi-square statistic:', round(chi2, 4))
print('p-value:', round(p, 6))
print('degrees of freedom:', dof)

# 5. 기대도수 확인
expected_df = pd.DataFrame(expected, index=table.index, columns=table.columns)

print('\n[기대도수]')
print(expected_df)

# 6. 채널별 구매전환율 (데이터 타입 에러 방지 포함)
df_convert = df.copy()
if df_convert['purchase_yn'].dtype == 'object':
    df_convert['purchase_yn'] = df_convert['purchase_yn'].str.strip().str.upper().map({'YES': 1, 'NO': 0, 'Y': 1, 'N': 0})

conversion_rate = df_convert.groupby('channel')['purchase_yn'].mean().sort_values(ascending=False) * 100

print('\n[채널별 구매전환율(%)]')
print(conversion_rate.round(2))

# 7. 해석 출력
print('\n[해석]')
if p < 0.05:
    print('p-value가 0.05보다 작으므로, 유입채널과 구매여부는 독립이 아니며 서로 관련이 있다고 해석할 수 있습니다.')
else:
    print('p-value가 0.05 이상이므로, 유입채널과 구매여부의 관련성을 확인하기 어렵습니다.')

print(f'가장 높은 구매전환율 채널은 {conversion_rate.index[0]}이며, 전환율은 {conversion_rate.iloc[0]:.2f}%입니다.')

[데이터 상위 5행]
  channel  purchase_yn
0    검색광고            1
1    검색광고            1
2    검색광고            1
3    검색광고            1
4    검색광고            1 

[변수 정보]
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 530 entries, 0 to 529
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   channel      530 non-null    object
 1   purchase_yn  530 non-null    int64 
dtypes: int64(1), object(1)
memory usage: 8.4+ KB
None 

[교차표]
purchase_yn   0    1
channel             
SNS          55   70
검색광고         50   90
이메일          80   45
직접방문         30  110 

[검정 결과]
chi-square statistic: 51.716
p-value: 0.0
degrees of freedom: 3

[기대도수]
purchase_yn          0          1
channel                          
SNS          50.707547  74.292453
검색광고         56.792453  83.207547
이메일          50.707547  74.292453
직접방문         56.792453  83.207547

[채널별 구매전환율(%)]
channel
직접방문    78.57
검색광고    64.29
SNS     56.00
이메일     36.00
Name: purchase_yn

In [9]:
import os
import zipfile
from pathlib import Path
import pandas as pd
from scipy.stats import chi2_contingency

if os.path.exists('type3_sample_csvs.zip'):
    with zipfile.ZipFile('type3_sample_csvs.zip', 'r') as z:
        z.extractall('/content/type3')

file_path = None
for root, dirs, files in os.walk('/content/type3'):
    if 'gender_preference.csv' in files:
        file_path = os.path.join(root, 'gender_preference.csv')
        break

df = pd.read_csv(file_path)

print('[데이터 상위 5행]')
print(df.head(), '\n')

print('[변수 정보]')
print(df.info(), '\n')

col_gender = 'gender'
col_content = 'content' if 'content' in df.columns else 'content_type'

table = pd.crosstab(df[col_gender], df[col_content])

print('[교차표]')
print(table, '\n')

chi2, p, dof, expected = chi2_contingency(table)

print('[검정 결과]')
print('chi-square statistic:', round(chi2, 4))
print('p-value:', round(p, 6))
print('degrees of freedom:', dof)

expected_df = pd.DataFrame(expected, index=table.index, columns=table.columns)

print('\n[기대도수]')
print(expected_df)

top_pref = table.idxmax(axis=1)

print('\n[성별별 최다 선호 유형]')
print(top_pref)

row_ratio = pd.crosstab(df[col_gender], df[col_content], normalize='index') * 100

print('\n[성별 내 콘텐츠 선호 비율(%)]')
print(row_ratio.round(2))

print

[데이터 상위 5행]
  gender content_type
0     남성        데이터분석
1     남성        데이터분석
2     남성        데이터분석
3     남성        데이터분석
4     남성        데이터분석 

[변수 정보]
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 330 entries, 0 to 329
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   gender        330 non-null    object
 1   content_type  330 non-null    object
dtypes: object(2)
memory usage: 5.3+ KB
None 

[교차표]
content_type  데이터분석  디자인  마케팅  프로그래밍
gender                              
남성               50   18   28     54
여성               42   48   55     35 

[검정 결과]
chi-square statistic: 24.6478
p-value: 1.8e-05
degrees of freedom: 3

[기대도수]
content_type      데이터분석   디자인        마케팅      프로그래밍
gender                                             
남성            41.818182  30.0  37.727273  40.454545
여성            50.181818  36.0  45.272727  48.545455

[성별별 최다 선호 유형]
gender
남성    프로그래밍
여성      마케팅
dtype: object

[성별 내 콘텐츠 선호 비율(

<function print(*args, sep=' ', end='\n', file=None, flush=False)>